In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Configurazione Percorsi
DATA_DIR = os.path.join("..", "data")
INPUT_ADDRESSES_FILE = os.path.join(DATA_DIR, "unique_spent_addresses.csv")

def get_file_names(start_date, end_date):
    date_dict = {}
    curr = start_date
    while curr <= end_date:
        name = f"Bitcoin_Data_{curr.strftime('%Y_%m_%d')}.csv"
        label = curr.strftime('%d %b %Y')
        date_dict[label] = name
        curr += timedelta(days=1)
    return date_dict

def load_global_inputs():
    if os.path.exists(INPUT_ADDRESSES_FILE):
        try:
            df_inputs = pd.read_csv(INPUT_ADDRESSES_FILE)
            return set(df_inputs.iloc[:, 0].astype(str))
        except Exception as e:
            print(f"Errore nel caricamento del file indirizzi: {e}")
            return set()
    return set()

global_input_set = load_global_inputs()
date_options = get_file_names(datetime(2024, 5, 24), datetime(2024, 6, 5))

# 3. Funzione di Analisi Aggiornata
def analyze_data(selected_files, min_in, max_in, min_out, max_out, unbal_threshold, 
                 noise_threshold, sum_threshold, 
                 use_chain_filter, use_io_filter, use_unbal_filter, use_noise_filter, use_sum_filter):
    
    total_stats = []

    for label, file_name in selected_files.items():
        file_path = os.path.join(DATA_DIR, file_name)
        if not os.path.exists(file_path):
            continue
            
        try:
            df = pd.read_csv(file_path)
            agg_funcs = {
                'output_value_BTC': list,
                'output_address': lambda x: set(x.dropna()),
                'transaction_inputs': 'first'
            }
            tx_df = df.groupby('transaction_hash').agg(agg_funcs).reset_index()
            initial_count = len(tx_df)
            tx_df['output_count'] = tx_df['output_address'].apply(len)
            
            # --- APPLICAZIONE FILTRI ---
            
            # FILTRO 1: Catena di spesa
            if use_chain_filter and global_input_set:
                tx_df = tx_df[tx_df['output_address'].apply(lambda s: not s.isdisjoint(global_input_set))]
            
            # FILTRO 2: Range Input/Output
            if use_io_filter:
                mask = (tx_df['transaction_inputs'] >= min_in) & (tx_df['transaction_inputs'] <= max_in) & \
                        (tx_df['output_count'] >= min_out) & (tx_df['output_count'] <= max_out)
                tx_df = tx_df[mask]
            
            # FILTRO 3: Unbalanced
            if use_unbal_filter:
                def check_ratio(v):
                    if len(v) < 2: return False
                    try:
                        return (max(v) / min(v)) >= unbal_threshold
                    except ZeroDivisionError: return True
                tx_df = tx_df[tx_df['output_value_BTC'].apply(check_ratio)]
            
            # FILTRO 4: Rumore (Soglia Minima di Valore su singolo output)
            if use_noise_filter:
                tx_df = tx_df[tx_df['output_value_BTC'].apply(lambda x: any(val >= noise_threshold for val in x))]
            
            # FILTRO 5: Volume Totale (Somma di tutti gli output della transazione)
            if use_sum_filter:
                tx_df = tx_df[tx_df['output_value_BTC'].apply(lambda x: sum(x) >= sum_threshold)]
            
            remaining = len(tx_df)
            total_stats.append({
                'Giorno': label, 
                'Iniziali': initial_count, 
                'Rimanenti': remaining, 
                'Scartate': initial_count - remaining
            })
        except Exception as e:
            print(f"Errore in {label}: {e}")
            
    return pd.DataFrame(total_stats)

# 4. Interfaccia Utente
style = {'description_width': 'initial'}

day_dropdown = widgets.Dropdown(options=[('Tutti i giorni', 'ALL')] + list(date_options.items()), value='ALL', description='Periodo:', style=style)

min_in_w = widgets.IntText(value=1, description='Min In:', layout=widgets.Layout(width='120px'))
max_in_w = widgets.IntText(value=10, description='Max In:', layout=widgets.Layout(width='120px'))
min_out_w = widgets.IntText(value=1, description='Min Out:', layout=widgets.Layout(width='120px'))
max_out_w = widgets.IntText(value=10, description='Max Out:', layout=widgets.Layout(width='120px'))

unbal_threshold_w = widgets.FloatText(value=10.0, description='Ratio Soglia:', layout=widgets.Layout(width='200px'))
noise_threshold_w = widgets.FloatText(value=0.1, description='Soglia BTC (almeno un output >=):', style=style, layout=widgets.Layout(width='300px'))
sum_threshold_w = widgets.FloatText(value=1.0, description='Soglia Somma BTC (Totale >=):', style=style, layout=widgets.Layout(width='300px'))

activate_chain = widgets.Checkbox(value=True, description='Filtro Catena (Spesa successiva)', style=style)
activate_io = widgets.Checkbox(value=True, description='Filtro Quantità In/Out', style=style)
activate_unbal = widgets.Checkbox(value=False, description='Filtro Unbalanced', style=style)
activate_noise = widgets.Checkbox(value=False, description='Filtro Rumore (Singolo Output)', style=style)
activate_sum = widgets.Checkbox(value=False, description='Filtro Volume Totale (Somma Output)', style=style)

btn = widgets.Button(description="Esegui Analisi", button_style='success', icon='filter')
output = widgets.Output()

def on_click(b):
    with output:
        clear_output()
        files = date_options if day_dropdown.value == 'ALL' else {k: v for k, v in date_options.items() if v == day_dropdown.value}
        res = analyze_data(files, min_in_w.value, max_in_w.value, min_out_w.value, max_out_w.value, 
                           unbal_threshold_w.value, noise_threshold_w.value, sum_threshold_w.value,
                           activate_chain.value, activate_io.value, activate_unbal.value, 
                           activate_noise.value, activate_sum.value)
        if not res.empty:
            display(res)
            print(f"\nTotale Rimanenti: {res['Rimanenti'].sum()} | Totale Scartate: {res['Scartate'].sum()}")
        else:
            print("Nessun risultato trovato.")

btn.on_click(on_click)

ui = widgets.VBox([
    widgets.HTML("<h2>Dashboard Analisi Transazioni Bitcoin</h2>"),
    day_dropdown,
    widgets.HTML("<hr><b>1. Catena di Spesa</b>"), activate_chain,
    widgets.HTML("<hr><b>2. Dimensioni (In/Out)</b>"), activate_io, widgets.HBox([min_in_w, max_in_w, min_out_w, max_out_w]),
    widgets.HTML("<hr><b>3. Sbilanciamento (Unbalanced)</b>"), activate_unbal, unbal_threshold_w,
    widgets.HTML("<hr><b>4. Soglia Valore Singolo (Filtro Rumore)</b>"), activate_noise, noise_threshold_w,
    widgets.HTML("<hr><b>5. Soglia Volume Totale (Somma di tutti gli Output)</b>"), activate_sum, sum_threshold_w,
    widgets.HTML("<br>"), btn, output
])
display(ui)